# CURE-Rec — remaining Phase D run

This notebook contains only the remaining executable reviewer action: paired user-level external ranking statistics. Phase A, B, C, CRN, and scalability work are intentionally not included and will not be rerun.

This notebook does not create causal policy evidence. It requires an audited per-user metrics file.

In [1]:
from pathlib import Path
import sys
import pandas as pd

CANDIDATES=[Path.cwd(),Path.cwd()/'paper-ideas'/'CURE-Rec'/'code',*Path.cwd().parents]
ROOT=next(p for p in CANDIDATES if (p/'pyproject.toml').exists() and (p/'cure_rec').exists())
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from cure_rec.revision_suite import paired_user_statistics
RESULTS=ROOT/'results'/'reviewer_phase_assets'
METRICS=RESULTS/'per_user_metrics.csv'
print('Metrics file:',METRICS)


Metrics file: /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code/results/reviewer_phase_assets/per_user_metrics.csv


## Generate the per-user file

This cell uses the registered chronological MovieLens evaluator and frozen evaluation protocol. It generates the missing input before running Phase D. It does not modify the CURE-Sim YAML configuration.


In [ ]:
from cure_rec.data import load_dataset
from cure_rec.models import chronological_leave_one_out, PopularityRecommender, BPRMFRecommender, evaluate_user_metrics

DATASET_SOURCE = ROOT / 'data' / 'raw' / 'ml-1m'
DOWNLOAD_MOVIELENS = True
MAX_EVAL_USERS = 1000
BPR_UPDATES = 500_000

if not METRICS.exists():
    dataset = load_dataset('movielens_1m', DATASET_SOURCE, download=DOWNLOAD_MOVIELENS)
    split = chronological_leave_one_out(dataset.interactions)
    popularity = PopularityRecommender().fit(split.train)
    bpr = BPRMFRecommender(max_updates=BPR_UPDATES, seed=42).fit(split.train)
    metrics = pd.concat([
        evaluate_user_metrics(popularity, split, k=10, max_users=MAX_EVAL_USERS),
        evaluate_user_metrics(bpr, split, k=10, max_users=MAX_EVAL_USERS),
    ], ignore_index=True)
    METRICS.parent.mkdir(parents=True, exist_ok=True)
    metrics.to_csv(METRICS, index=False)
    print('Generated:', METRICS, 'rows:', len(metrics))
else:
    print('Using existing:', METRICS)


## Required input

The input must contain one row per user-model pair with these columns:

```text
user_id,model,hit,ndcg
```

Do not construct this file from aggregate metrics.

In [2]:
if not METRICS.exists():
    print('Phase D is not runnable yet: per_user_metrics.csv is missing.')
    print('No unsupported statistics were generated.')
else:
    metrics=pd.read_csv(METRICS)
    required={'user_id','model','hit','ndcg'}
    missing=required-set(metrics.columns)
    if missing: raise ValueError(f'Missing required columns: {sorted(missing)}')
    print('Rows:',len(metrics),'Users:',metrics.user_id.nunique(),'Models:',metrics.model.unique().tolist())
    ci,tests=paired_user_statistics(metrics)
    ci.to_csv(RESULTS/'paired_bootstrap_ci.csv',index=False)
    tests.to_csv(RESULTS/'paired_tests_holm.csv',index=False)
    display(ci)
    display(tests)

Phase D is not runnable yet: per_user_metrics.csv is missing.
No unsupported statistics were generated.


## Interpretation and scope

Report paired bootstrap intervals, effect sizes, raw sign-test/permutation p-values, and Holm-adjusted p-values. These are external chronological-ranking statistics only. They do not establish long-term causal policy effects or validate CURE-Sim interventions.